# 06 · Evaluation, error analysis and the comparison

**ITRI626 · Colile Sibanda**

Objectives **O6**, **O7** and **O8**. Every number quoted in the report
and the presentation comes from this notebook's outputs, and every one of
them appears in `outputs/metrics/comparison.csv`.

This is the only notebook that touches the held-out split.


In [ ]:
# Every notebook opens with the same four lines: path, config, seed, paths.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from docclf.config.loader import load_config
from docclf.config.paths import resolve_paths
from docclf.config.seed import set_global_seed
from docclf.utils.env import detect_environment

CONFIG = ROOT / "configs" / "base.yaml"
ENV = detect_environment()
cfg = load_config(CONFIG, env=ENV)
paths = resolve_paths(cfg, ENV).ensure()
set_global_seed(cfg.seed)

f"Config: {cfg.name} | Environment: {ENV} | Seed: {cfg.seed}"

In [ ]:
MODELS = ['baseline_cnn', 'resnet50', 'vit_b16']  # update if a different backbone was chosen
MODELS

## Load the committed split and re-verify it


In [ ]:
from docclf.splits.io import load_splits
from docclf.splits.leakage import assert_no_group_overlap
from docclf.data.loaders import build_dataloaders
from docclf.utils.env import select_device

splits = load_splits(paths.splits)
assert_no_group_overlap(splits, cfg.split.group_column)
loaders = build_dataloaders(splits, cfg)
device = select_device(cfg)
print(f'Held-out pages: {len(loaders["test"].dataset)}')
print(f'Device: {device}')

## Predict with each trained model

Each model is restored from its best checkpoint and run over the same held-out loader.


In [ ]:
import dataclasses
from docclf.models.registry import build_model
from docclf.training.callbacks import Checkpoint
from docclf.evaluation.predict import collect_predictions
from docclf.evaluation.metrics import compute_metrics
from docclf.evaluation.efficiency import measure_inference

predictions, metrics, efficiency = {}, {}, {}
for name in MODELS:
    run_cfg = dataclasses.replace(cfg, train=dataclasses.replace(cfg.train, model=name))
    model = build_model(name, cfg.n_classes, run_cfg)
    Checkpoint(paths.checkpoints, name).load_best(model, map_location=device)
    predictions[name] = collect_predictions(model, loaders['test'], device, cfg.classes)
    metrics[name] = compute_metrics(predictions[name], name)
    efficiency[name] = measure_inference(model, loaders['test'], device, name, repeats=cfg.eval.latency_repeats)
    metrics[name].save(paths.metrics)
    predictions[name].save(paths.metrics, name)
    print(f'{name}: accuracy {metrics[name].accuracy:.4f} | macro-F1 {metrics[name].macro_f1:.4f}')

## The comparison table

All three models, one held-out split. `comparison.csv` is the single source of every number in the report.


In [ ]:
from docclf.evaluation.compare import build_comparison_table, save_comparison, recommend
from docclf.models.registry import model_family

families = {name: model_family(name) for name in MODELS}
comparison = build_comparison_table(metrics, efficiency, families)
save_comparison(comparison, paths.metrics)
comparison

## Per-class performance


In [ ]:
import pandas as pd

per_class = pd.concat(
    [metrics[name].per_class.assign(model=name) for name in MODELS], ignore_index=True
)
pivot = per_class.pivot(index='class_name', columns='model', values='f1').reindex(cfg.classes)
ax = pivot.plot.bar(figsize=(11, 4))
ax.set_ylabel('F1')
ax.set_ylim(0, 1)
ax.set_title('Per-class F1 on the held-out split')
pivot.round(4)

## Confusion matrices

Rows are true classes, columns predicted — stated on the figure itself.


In [ ]:
from docclf.evaluation.confusion import confusion_frame, most_confused_pairs, plot_confusion
from IPython.display import Image as Show, display

for name in MODELS:
    path = plot_confusion(predictions[name], paths.figures / f'confusion_{name}.png', name)
    plot_confusion(predictions[name], paths.figures / f'confusion_normalised_{name}.png', name, normalise=True)
    display(Show(str(path)))

## Error analysis (objective O7)

Which pairs are confused, in which direction, and what the pages look like.


In [ ]:
pairs = pd.concat(
    [most_confused_pairs(predictions[name], top_n=5).assign(model=name) for name in MODELS],
    ignore_index=True,
)
pairs.to_csv(paths.metrics / 'confused_pairs.csv', index=False)
pairs

In [ ]:
from docclf.evaluation.errors import plot_error_examples, sample_errors

best_model = comparison.iloc[0]['model']
path = plot_error_examples(
    predictions[best_model], paths.figures / f'errors_{best_model}.png', cfg.eval.error_samples, best_model
)
display(Show(str(path)))
sample_errors(predictions[best_model], cfg.eval.error_samples)

## Efficiency and the recommendation (objective O8)

Accuracy is only half the argument. State the latency, the size, the
device they were measured on, and the conditions under which the
recommendation would change.


In [ ]:
efficiency_table = pd.DataFrame([efficiency[name].as_row() for name in MODELS])
efficiency_table.to_csv(paths.metrics / 'efficiency.csv', index=False)
efficiency_table

In [ ]:
print(recommend(comparison, cfg.eval.review_threshold))
print()
print('The recommendation changes if: the intake queue needs a lower per-page')
print('latency than the leading model provides; or the deployment target has')
print('no GPU, which moves the argument towards the smaller model.')